# 🚂 RailSafe — Feature Engineering + Entraînement XGBoost

**Objectif** : Construire le dataset ML, entraîner XGBoost, évaluer avec SHAP.

**Pipeline :**
1. Feature engineering sur TGV, TER, Intercités
2. Fusion en un dataset unifié
3. Entraînement XGBoost (classification binaire : retard élevé ou non)
4. Évaluation + SHAP
5. Sauvegarde du modèle

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from xgboost import XGBClassifier

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

RAW_DIR      = Path('../data/raw')
PROC_DIR     = Path('../data/processed')
MODELS_DIR   = Path('../models')
PROC_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print('✅ Imports OK')

## 2. Feature Engineering — TGV

In [ ]:
tgv = pd.read_csv(RAW_DIR / 'regularite_tgv.csv', sep=';', encoding='utf-8')

# Parsing date
tgv['date']  = pd.to_datetime(tgv['Date'], format='%Y-%m')
tgv['annee'] = tgv['date'].dt.year
tgv['mois']  = tgv['date'].dt.month
tgv['trimestre'] = tgv['date'].dt.quarter

# Features métier
tgv['taux_retard']     = tgv["Nombre de trains en retard à l'arrivée"] / tgv['Nombre de circulations prévues']
tgv['taux_annulation'] = tgv['Nombre de trains annulés'] / tgv['Nombre de circulations prévues']
tgv['taux_retard_30m'] = tgv['Nombre trains en retard > 30min'] / tgv['Nombre de circulations prévues']
tgv['taux_retard_60m'] = tgv['Nombre trains en retard > 60min'] / tgv['Nombre de circulations prévues']

# Liaison
tgv['liaison'] = tgv['Gare de départ'].str.strip() + ' -> ' + tgv["Gare d'arrivée"].str.strip()

# Nettoyage anomalies (liaisons avec gare inconnue)
tgv = tgv[tgv["Gare d'arrivée"].notna()]
tgv = tgv[tgv["Gare d'arrivée"] != '0']
tgv = tgv[tgv['Nombre de circulations prévues'] > 0]

# Cible binaire
seuil_tgv = tgv['taux_retard'].quantile(0.75)
tgv['retard_eleve'] = (tgv['taux_retard'] > seuil_tgv).astype(int)

# Type de service
tgv['type_service'] = 'TGV'
tgv['region']       = 'National'

print(f'TGV après nettoyage : {tgv.shape[0]:,} lignes')
print(f'Seuil retard élevé TGV : {seuil_tgv*100:.1f}%')

## 3. Feature Engineering — TER

In [ ]:
ter = pd.read_csv(RAW_DIR / 'regularite_ter.csv', sep=';', encoding='utf-8')

ter['date']      = pd.to_datetime(ter['Date'], format='%Y-%m')
ter['annee']     = ter['date'].dt.year
ter['mois']      = ter['date'].dt.month
ter['trimestre'] = ter['date'].dt.quarter

ter['taux_retard']     = ter["Nombre de trains en retard à l'arrivée"] / ter['Nombre de trains programmés']
ter['taux_annulation'] = ter['Nombre de trains annulés'] / ter['Nombre de trains programmés']
ter['taux_retard_30m'] = np.nan
ter['taux_retard_60m'] = np.nan

ter['liaison']      = 'TER_' + ter['Région'].str.strip()
ter['type_service'] = 'TER'
ter['region']       = ter['Région']

# Causes retards (non disponibles pour TER)
for col in ['Prct retard pour causes externes',
            'Prct retard pour cause infrastructure',
            'Prct retard pour cause gestion trafic',
            'Prct retard pour cause matériel roulant']:
    ter[col] = np.nan

# Nettoyage
ter = ter[ter['Nombre de trains programmés'] > 0]
ter = ter[ter['taux_retard'].notna()]

seuil_ter = ter['taux_retard'].quantile(0.75)
ter['retard_eleve'] = (ter['taux_retard'] > seuil_ter).astype(int)

print(f'TER après nettoyage : {ter.shape[0]:,} lignes')
print(f'Seuil retard élevé TER : {seuil_ter*100:.1f}%')

## 4. Feature Engineering — Intercités

In [ ]:
ic = pd.read_csv(RAW_DIR / 'regularite_intercites.csv', sep=';', encoding='utf-8')

ic['date']      = pd.to_datetime(ic['Date'], format='%Y-%m')
ic['annee']     = ic['date'].dt.year
ic['mois']      = ic['date'].dt.month
ic['trimestre'] = ic['date'].dt.quarter

ic['taux_retard']     = ic["Nombre de trains en retard à l'arrivée"] / ic['Nombre de trains programmés']
ic['taux_annulation'] = ic['Nombre de trains annulés'] / ic['Nombre de trains programmés']
ic['taux_retard_30m'] = np.nan
ic['taux_retard_60m'] = np.nan

ic['liaison']      = ic['Départ'].str.strip() + ' -> ' + ic['Arrivée'].str.strip()
ic['type_service'] = 'IC'
ic['region']       = 'National'

for col in ['Prct retard pour causes externes',
            'Prct retard pour cause infrastructure',
            'Prct retard pour cause gestion trafic',
            'Prct retard pour cause matériel roulant']:
    ic[col] = np.nan

ic = ic[ic['Nombre de trains programmés'] > 0]
ic = ic[ic['taux_retard'].notna()]

seuil_ic = ic['taux_retard'].quantile(0.75)
ic['retard_eleve'] = (ic['taux_retard'] > seuil_ic).astype(int)

print(f'Intercités après nettoyage : {ic.shape[0]:,} lignes')
print(f'Seuil retard élevé IC : {seuil_ic*100:.1f}%')

## 5. Fusion du dataset unifié

In [ ]:
FEATURES = [
    'annee', 'mois', 'trimestre',
    'taux_annulation', 'taux_retard_30m', 'taux_retard_60m',
    'Prct retard pour causes externes',
    'Prct retard pour cause infrastructure',
    'Prct retard pour cause gestion trafic',
    'Prct retard pour cause matériel roulant',
    'type_service', 'region', 'liaison',
    'retard_eleve'
]

def select_features(df):
    available = [c for c in FEATURES if c in df.columns]
    return df[available].copy()

df = pd.concat([
    select_features(tgv),
    select_features(ter),
    select_features(ic)
], ignore_index=True)

print(f'Dataset unifié : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'\nDistribution cible :')
print(df['retard_eleve'].value_counts())
print(f'\nValeurs manquantes :')
print(df.isnull().sum())

## 6. Encodage & Préparation ML

In [ ]:
df_ml = df.copy()

# Encodage des variables catégorielles
le_type    = LabelEncoder()
le_region  = LabelEncoder()
le_liaison = LabelEncoder()

df_ml['type_service_enc'] = le_type.fit_transform(df_ml['type_service'].astype(str))
df_ml['region_enc']       = le_region.fit_transform(df_ml['region'].fillna('Unknown').astype(str))
df_ml['liaison_enc']      = le_liaison.fit_transform(df_ml['liaison'].fillna('Unknown').astype(str))

# Features finales pour XGBoost
FEATURE_COLS = [
    'annee', 'mois', 'trimestre',
    'taux_annulation',
    'Prct retard pour causes externes',
    'Prct retard pour cause infrastructure',
    'Prct retard pour cause gestion trafic',
    'Prct retard pour cause matériel roulant',
    'type_service_enc', 'region_enc', 'liaison_enc'
]

TARGET = 'retard_eleve'

X = df_ml[FEATURE_COLS]
y = df_ml[TARGET]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'\nFeatures : {FEATURE_COLS}')

## 7. Entraînement XGBoost

In [ ]:
model = XGBClassifier(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 3,
    random_state     = 42,
    eval_metric      = 'logloss',
    early_stopping_rounds = 20,
    verbosity        = 0
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print(f'✅ Entraînement terminé')
print(f'Meilleure itération : {model.best_iteration}')

## 8. Évaluation

In [ ]:
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

auc   = roc_auc_score(y_test, y_pred_prob)
f1    = f1_score(y_test, y_pred)

print('=' * 55)
print('📊 RÉSULTATS XGBoost')
print('=' * 55)
print(f'ROC-AUC : {auc:.4f}')
print(f'F1-Score : {f1:.4f}')
print()
print(classification_report(y_test, y_pred,
      target_names=['Normal', 'Retard élevé']))

In [ ]:
# Courbe ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC AUC = {auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('Taux de faux positifs')
axes[0].set_ylabel('Taux de vrais positifs')
axes[0].set_title('Courbe ROC — XGBoost', fontweight='bold')
axes[0].legend()

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Normal', 'Retard élevé'],
            yticklabels=['Normal', 'Retard élevé'])
axes[1].set_title('Matrice de confusion', fontweight='bold')
axes[1].set_ylabel('Réel')
axes[1].set_xlabel('Prédit')

plt.tight_layout()
plt.savefig('../data/viz_roc_confusion.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Sauvegardé : data/viz_roc_confusion.png')

## 9. SHAP — Explicabilité

In [ ]:
# SHAP values
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Noms lisibles
feature_names_fr = {
    'annee'                                          : 'Année',
    'mois'                                           : 'Mois',
    'trimestre'                                      : 'Trimestre',
    'taux_annulation'                                : 'Taux annulation',
    'Prct retard pour causes externes'               : 'Cause externe (%)',
    'Prct retard pour cause infrastructure'          : 'Infrastructure (%)',
    'Prct retard pour cause gestion trafic'          : 'Gestion trafic (%)',
    'Prct retard pour cause matériel roulant'        : 'Matériel roulant (%)',
    'type_service_enc'                               : 'Type service',
    'region_enc'                                     : 'Région',
    'liaison_enc'                                    : 'Liaison',
}
X_test_named = X_test.rename(columns=feature_names_fr)

fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test_named,
    plot_type='bar',
    show=False,
    color='steelblue'
)
plt.title('SHAP — Importance des features (XGBoost)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../data/viz_shap_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Sauvegardé : data/viz_shap_importance.png')

In [ ]:
# SHAP beeswarm (impact directionnel)
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test_named,
    show=False
)
plt.title('SHAP — Impact directionnel des features', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../data/viz_shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Sauvegarde du modèle

In [ ]:
# Sauvegarde modèle + encodeurs + métadonnées
model_bundle = {
    'model'        : model,
    'feature_cols' : FEATURE_COLS,
    'le_type'      : le_type,
    'le_region'    : le_region,
    'le_liaison'   : le_liaison,
    'seuil_tgv'    : seuil_tgv,
    'seuil_ter'    : seuil_ter,
    'seuil_ic'     : seuil_ic,
    'metrics'      : {
        'roc_auc' : round(auc, 4),
        'f1'      : round(f1, 4),
    }
}

joblib.dump(model_bundle, MODELS_DIR / 'railsafe_model.joblib')
print(f'✅ Modèle sauvegardé : models/railsafe_model.joblib')

# Sauvegarde dataset ML
df_ml.to_parquet(PROC_DIR / 'dataset_ml.parquet', index=False)
print(f'✅ Dataset ML sauvegardé : data/processed/dataset_ml.parquet')

print(f"""
{'='*55}
📊 RÉSUMÉ FINAL
{'='*55}
Modèle    : XGBoost (n={model.best_iteration} arbres)
ROC-AUC   : {auc:.4f}
F1-Score  : {f1:.4f}
Features  : {len(FEATURE_COLS)}
Train     : {X_train.shape[0]:,} lignes
Test      : {X_test.shape[0]:,} lignes

➡️  Prochaine étape : src/model.py + api/main.py
""")